<a href="https://colab.research.google.com/github/korkutanapa/OEE_ARTICLE_STUDIES/blob/main/TS_TDA_PREDICTION_TDA_STEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q "numpy==1.26.4" "scipy==1.11.4" "scikit-learn==1.3.2"
%pip install -q "giotto-tda==0.6.2"

In [ ]:
# Install feature_engine (run this once in a cell)
%pip install feature_engine

In [3]:
import pandas as pd
# Load the file for time series
data_path = 'tda_ready.xlsx'
df = pd.read_excel(data_path)

In [4]:
window_length = 24
window_size = 24
time_delay = 8
dimension = 3
stride = 1

In [5]:
import numpy as np
from gtda.time_series import SlidingWindow, TakensEmbedding
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import Scaler

# --- 1) Extract the time series as (n_timestamps, 1) ---
# df: your DataFrame, with a column 'Operation'
X = df['Operation'].to_numpy(dtype=float).reshape(-1, 1)   # shape: (n_timestamps, 1)

# --- 2) Sliding window: cut into overlapping windows ---
SW = SlidingWindow(size=window_size, stride=stride)
# Typical output shape: (n_windows, window_size, n_features)
X_sw = SW.fit_transform(X)

# For convenience, treat each window as an independent univariate series
n_windows, win_len, n_feat = X_sw.shape  # n_feat should be 1
X_windows = X_sw.reshape(n_windows, win_len)   # shape: (n_windows, window_size)

# --- 3) Takens embedding on each window ---
TE = TakensEmbedding(time_delay=time_delay, dimension=dimension)
# Input: (n_windows, window_size)
# Output: (n_windows, n_points, dimension)
X_te = TE.fit_transform(X_windows)

# --- 4) Vietoris–Rips persistence on embedded windows ---
VR = VietorisRipsPersistence(homology_dimensions=[0, 1])
# Input: (n_windows, n_points, dimension)
# Output: array of diagrams, one per window
diagrams = VR.fit_transform(X_te)

# --- 5) Scale persistence diagrams (normalization in diagram space) ---
scaler = Scaler()
scaled_diagrams = scaler.fit_transform(diagrams)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


new tda features version 2025 december

In [6]:
import numpy as np
import pandas as pd
from gtda.diagrams import (
    PersistenceEntropy,
    Amplitude,
    BettiCurve,
    PersistenceLandscape,
    Silhouette,
    HeatKernel,
)
from scipy.stats import skew, kurtosis

# -------------------------------------------------------------------
# Small helpers
# -------------------------------------------------------------------
def safe_skew(x):
    if x.size < 2 or np.allclose(x, x[0]):
        return 0.0
    return float(skew(x, bias=False))

def safe_kurtosis(x):
    if x.size < 2 or np.allclose(x, x[0]):
        return 0.0
    return float(kurtosis(x, bias=False, fisher=True))

def safe_gini(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return 0.0
    x = np.abs(x)
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    cum = np.cumsum(x)
    g = (n + 1 - 2.0 * np.sum(cum) / cum[-1]) / n
    return float(g)

def safe_quantile(x, q):
    if x.size == 0:
        return 0.0
    return float(np.quantile(x, q))

def safe_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 2 or y.size < 2:
        return 0.0
    if np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

def safe_ratio(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(den != 0, num / den, 0.0)
    return r

# -------------------------------------------------------------------
# Helper: Carlsson features
# -------------------------------------------------------------------
def compute_carlsson_features(births, deaths, fn_max=5):
    """Compute Carlsson-type features f1..f5 for a single diagram."""
    if len(births) == 0:
        return [0.0] * fn_max

    lifetimes = deaths - births
    d_max = np.max(deaths) if len(deaths) > 0 else 0.0

    f1 = np.sum(births * lifetimes) if fn_max >= 1 else None
    f2 = np.sum((d_max - deaths) * lifetimes) if fn_max >= 2 else None
    f3 = np.sum((births ** 2) * (lifetimes ** 4)) if fn_max >= 3 else None
    f4 = np.sum(((d_max - deaths) ** 2) * (lifetimes ** 4)) if fn_max >= 4 else None
    f5 = np.max(lifetimes) if (fn_max >= 5 and len(lifetimes) > 0) else None

    values = [f1, f2, f3, f4, f5][:fn_max]
    return [v if v is not None else 0.0 for v in values]


# -------------------------------------------------------------------
# Helper: Heat kernel feature extraction (compressed, rich)
# -------------------------------------------------------------------
def extract_heat_kernel_features(heat_kernel_features, eps=1e-6, max_singular_values=3):
    """
    heat_kernel_features: array of shape
        (n_windows, 2, n_bins, n_bins) for H0 and H1

    Returns:
        DataFrame (n_windows, many_features)
    """
    feature_list = []

    for i in range(heat_kernel_features.shape[0]):  # per window
        sample_feats = {}

        for dim_idx, hom_dim in enumerate(["H0", "H1"]):
            matrix = heat_kernel_features[i, dim_idx, :, :]  # shape: (n_bins, n_bins)
            vals = matrix.flatten()
            abs_vals = np.abs(vals)

            # Global L1/L2
            L1 = float(np.sum(abs_vals))
            L2 = float(np.sqrt(np.sum(vals ** 2)))
            sample_feats[f"HK_L1_{hom_dim}"] = L1
            sample_feats[f"HK_L2_{hom_dim}"] = L2

            # Basic stats
            sample_feats[f"HK_mean_{hom_dim}"] = float(np.mean(vals)) if vals.size > 0 else 0.0
            sample_feats[f"HK_median_{hom_dim}"] = float(np.median(vals)) if vals.size > 0 else 0.0
            sample_feats[f"HK_std_{hom_dim}"] = float(np.std(vals)) if vals.size > 0 else 0.0
            sample_feats[f"HK_var_{hom_dim}"] = float(np.var(vals)) if vals.size > 0 else 0.0
            sample_feats[f"HK_skew_{hom_dim}"] = safe_skew(vals)
            sample_feats[f"HK_kurt_{hom_dim}"] = safe_kurtosis(vals)

            # Energy (Frobenius^2) and sparsity
            energy = float(np.sum(vals ** 2))
            sample_feats[f"HK_energy_{hom_dim}"] = energy
            sparsity_ratio = float(np.mean(abs_vals < eps)) if vals.size > 0 else 0.0
            sample_feats[f"HK_sparsity_ratio_{hom_dim}"] = sparsity_ratio

            # Entropy (using |vals| normalized as probabilities)
            if L1 > 0:
                p = abs_vals / L1
                # avoid log(0)
                p_safe = p[p > 0]
                entropy_val = -float(np.sum(p_safe * np.log(p_safe)))
            else:
                entropy_val = 0.0
            sample_feats[f"HK_entropy_{hom_dim}"] = entropy_val

            # Row/column aggregates (using L1 per row/col)
            row_sums = np.sum(abs_vals.reshape(matrix.shape), axis=1)
            col_sums = np.sum(abs_vals.reshape(matrix.shape), axis=0)

            for label, arr in [("row", row_sums), ("col", col_sums)]:
                if arr.size > 0:
                    sample_feats[f"HK_{label}_L1_mean_{hom_dim}"] = float(np.mean(arr))
                    sample_feats[f"HK_{label}_L1_std_{hom_dim}"] = float(np.std(arr))
                    sample_feats[f"HK_{label}_L1_min_{hom_dim}"] = float(np.min(arr))
                    sample_feats[f"HK_{label}_L1_max_{hom_dim}"] = float(np.max(arr))
                else:
                    sample_feats[f"HK_{label}_L1_mean_{hom_dim}"] = 0.0
                    sample_feats[f"HK_{label}_L1_std_{hom_dim}"] = 0.0
                    sample_feats[f"HK_{label}_L1_min_{hom_dim}"] = 0.0
                    sample_feats[f"HK_{label}_L1_max_{hom_dim}"] = 0.0

            # Symmetry measure: ||HK - HK^T||_F
            sym_diff = matrix - matrix.T
            sym_F = float(np.sqrt(np.sum(sym_diff ** 2)))
            sample_feats[f"HK_symmetry_F_{hom_dim}"] = sym_F

            # SVD singular values (PCA-like compression)
            try:
                s = np.linalg.svd(matrix, compute_uv=False)
                for k in range(max_singular_values):
                    val = float(s[k]) if k < len(s) else 0.0
                    sample_feats[f"HK_SV{k+1}_{hom_dim}"] = val
            except np.linalg.LinAlgError:
                for k in range(max_singular_values):
                    sample_feats[f"HK_SV{k+1}_{hom_dim}"] = 0.0

            # Graph Laplacian spectral features (on |HK|)
            w = np.abs(matrix)
            deg = np.sum(w, axis=1)
            L = np.diag(deg) - w
            try:
                eigvals = np.linalg.eigvalsh(L)
                eigvals = np.real(eigvals)
                if eigvals.size > 0:
                    eig_min = float(np.min(eigvals))
                    eig_max = float(np.max(eigvals))
                    trace_L = float(np.trace(L))
                    fro_L = float(np.sqrt(np.sum(L ** 2)))
                else:
                    eig_min = eig_max = trace_L = fro_L = 0.0
            except np.linalg.LinAlgError:
                eig_min = eig_max = trace_L = fro_L = 0.0

            sample_feats[f"HK_Laplacian_eig_min_{hom_dim}"] = eig_min
            sample_feats[f"HK_Laplacian_eig_max_{hom_dim}"] = eig_max
            sample_feats[f"HK_Laplacian_trace_{hom_dim}"] = trace_L
            sample_feats[f"HK_Laplacian_fro_{hom_dim}"] = fro_L

        feature_list.append(sample_feats)

    heat_kernel_df = pd.DataFrame(feature_list)
    return heat_kernel_df


# -------------------------------------------------------------------
# Helper: Betti / landscape / silhouette full features
# -------------------------------------------------------------------
def compute_curve_features(curve_array, prefix):
    """
    curve_array: shape (n_windows, 2, n_points)   (H0, H1)
    prefix: string prefix for column names (e.g. 'betti', 'landscape', 'silhouette')

    Returns DataFrame with columns:
        L1, L2, max, mean, median, std, var, skew, kurt,
        AUC, gini, nonzero_count, sparsity, for H0 and H1.
    """
    n_windows = curve_array.shape[0]

    feats = {}

    for hom_idx, hom_dim in enumerate(["H0", "H1"]):
        feats[f"{prefix}_L1_{hom_dim}"] = []
        feats[f"{prefix}_L2_{hom_dim}"] = []
        feats[f"{prefix}_max_{hom_dim}"] = []
        feats[f"{prefix}_mean_{hom_dim}"] = []
        feats[f"{prefix}_median_{hom_dim}"] = []
        feats[f"{prefix}_std_{hom_dim}"] = []
        feats[f"{prefix}_var_{hom_dim}"] = []
        feats[f"{prefix}_skew_{hom_dim}"] = []
        feats[f"{prefix}_kurt_{hom_dim}"] = []
        feats[f"{prefix}_AUC_{hom_dim}"] = []
        feats[f"{prefix}_gini_{hom_dim}"] = []
        feats[f"{prefix}_nonzero_count_{hom_dim}"] = []
        feats[f"{prefix}_sparsity_{hom_dim}"] = []

    for i in range(n_windows):
        for hom_idx, hom_dim in enumerate(["H0", "H1"]):
            v = curve_array[i, hom_idx, :]
            v = np.asarray(v, dtype=float)
            if v.size == 0:
                L1 = L2 = vmax = vmean = vmed = vstd = vvar = vaskew = vakurt = 0.0
                auc = g = 0.0
                nonzero = 0
                sparsity = 0.0
            else:
                L1 = float(np.sum(np.abs(v)))
                L2 = float(np.sqrt(np.sum(v ** 2)))
                vmax = float(np.max(v))
                vmean = float(np.mean(v))
                vmed = float(np.median(v))
                vstd = float(np.std(v))
                vvar = float(np.var(v))
                vaskew = safe_skew(v)
                vakurt = safe_kurtosis(v)
                # AUC ~ integral over discrete grid
                auc = float(np.trapz(v))
                g = safe_gini(v)
                nonzero = int(np.count_nonzero(v))
                sparsity = float(1.0 - nonzero / v.size)

            feats[f"{prefix}_L1_{hom_dim}"].append(L1)
            feats[f"{prefix}_L2_{hom_dim}"].append(L2)
            feats[f"{prefix}_max_{hom_dim}"].append(vmax)
            feats[f"{prefix}_mean_{hom_dim}"].append(vmean)
            feats[f"{prefix}_median_{hom_dim}"].append(vmed)
            feats[f"{prefix}_std_{hom_dim}"].append(vstd)
            feats[f"{prefix}_var_{hom_dim}"].append(vvar)
            feats[f"{prefix}_skew_{hom_dim}"].append(vaskew)
            feats[f"{prefix}_kurt_{hom_dim}"].append(vakurt)
            feats[f"{prefix}_AUC_{hom_dim}"].append(auc)
            feats[f"{prefix}_gini_{hom_dim}"].append(g)
            feats[f"{prefix}_nonzero_count_{hom_dim}"].append(nonzero)
            feats[f"{prefix}_sparsity_{hom_dim}"].append(sparsity)

    df = pd.DataFrame(feats)
    return df


# -------------------------------------------------------------------
# Helper: flatten simple vector features
# -------------------------------------------------------------------
def flatten_features_to_df(features_array, prefix):
    """
    features_array: np.ndarray of shape (n_windows, ...)
    prefix: base name for columns
    """
    flattened = features_array.reshape(features_array.shape[0], -1)
    cols = [f"{prefix}_{i}" for i in range(flattened.shape[1])]
    return pd.DataFrame(flattened, columns=cols)


# -------------------------------------------------------------------
# Helper: diagram statistics + Carlsson features + shape features
# -------------------------------------------------------------------
def extract_diagram_statistics(scaled_diagrams, fn_max=5):
    """
    scaled_diagrams: array-like of diagrams, each of shape (n_points, 3)
                     columns: [birth, death, homology_dim]
    Returns:
        features_df: DataFrame with per-window summary stats + Carlsson features.
    """
    n_windows = len(scaled_diagrams)
    features = {}

    for dim in [0, 1]:  # H0, H1
        # filter diagrams by dimension
        filtered_diagrams = [
            diagram[diagram[:, 2] == dim] for diagram in scaled_diagrams
        ]

        for i, diagram_dim in enumerate(filtered_diagrams):
            lifetimes = (
                diagram_dim[:, 1] - diagram_dim[:, 0]
                if len(diagram_dim) > 0
                else np.array([])
            )
            births = diagram_dim[:, 0] if len(diagram_dim) > 0 else np.array([])
            deaths = diagram_dim[:, 1] if len(diagram_dim) > 0 else np.array([])

            sample_key = f"sample_{i}"

            if sample_key not in features:
                features[sample_key] = {}

            # Basic counts
            features[sample_key][f"dim_{dim}_num_features"] = int(len(lifetimes))
            features[sample_key][f"dim_{dim}_sum_lifetimes"] = float(np.sum(lifetimes))

            if len(lifetimes) > 0:
                # Lifetime statistics
                max_life = float(np.max(lifetimes))
                mean_life = float(np.mean(lifetimes))
                median_life = float(np.median(lifetimes))
                std_life = float(np.std(lifetimes))
                var_life = float(np.var(lifetimes))
                min_life = float(np.min(lifetimes))

                features[sample_key][f"dim_{dim}_max_lifetime"] = max_life
                features[sample_key][f"dim_{dim}_mean_lifetime"] = mean_life
                features[sample_key][f"dim_{dim}_median_lifetime"] = median_life
                features[sample_key][f"dim_{dim}_std_lifetime"] = std_life
                features[sample_key][f"dim_{dim}_variance_lifetime"] = var_life
                features[sample_key][f"dim_{dim}_min_lifetime"] = min_life

                # Lifetime shape: skew, kurtosis, IQR, CV, Gini, tail, top-k
                features[sample_key][f"dim_{dim}_skew_lifetime"] = safe_skew(lifetimes)
                features[sample_key][f"dim_{dim}_kurt_lifetime"] = safe_kurtosis(lifetimes)
                features[sample_key][f"dim_{dim}_IQR_lifetime"] = float(
                    safe_quantile(lifetimes, 0.75) - safe_quantile(lifetimes, 0.25)
                )
                features[sample_key][f"dim_{dim}_CV_lifetime"] = (
                    float(std_life / mean_life) if mean_life != 0 else 0.0
                )
                features[sample_key][f"dim_{dim}_Gini_lifetime"] = safe_gini(lifetimes)
                features[sample_key][f"dim_{dim}_Q90_lifetime"] = safe_quantile(
                    lifetimes, 0.90
                )
                features[sample_key][f"dim_{dim}_Q95_lifetime"] = safe_quantile(
                    lifetimes, 0.95
                )
                features[sample_key][f"dim_{dim}_Q99_lifetime"] = safe_quantile(
                    lifetimes, 0.99
                )

                # Top-k lifetimes and shares
                sorted_life = np.sort(lifetimes)[::-1]
                for k in [1, 3, 5]:
                    if len(sorted_life) >= k:
                        topk = sorted_life[:k]
                        topk_sum = float(np.sum(topk))
                        features[sample_key][f"dim_{dim}_top{k}_lifetime_sum"] = topk_sum
                        features[sample_key][f"dim_{dim}_top{k}_lifetime_share"] = (
                            float(topk_sum / np.sum(lifetimes))
                            if np.sum(lifetimes) != 0
                            else 0.0
                        )
                    else:
                        features[sample_key][f"dim_{dim}_top{k}_lifetime_sum"] = 0.0
                        features[sample_key][f"dim_{dim}_top{k}_lifetime_share"] = 0.0

                # Persistence ratios
                sum_life = float(np.sum(lifetimes))
                max_ratio = float(max_life / sum_life) if sum_life != 0 else 0.0
                features[sample_key][f"dim_{dim}_max_lifetime_ratio"] = max_ratio

                # Birth/death statistics
                features[sample_key][f"dim_{dim}_sum_birth_times"] = float(np.sum(births))
                features[sample_key][f"dim_{dim}_mean_birth_time"] = float(np.mean(births))
                features[sample_key][f"dim_{dim}_sum_death_times"] = float(np.sum(deaths))
                features[sample_key][f"dim_{dim}_mean_death_time"] = float(np.mean(deaths))

                # Birth/death shape
                features[sample_key][f"dim_{dim}_range_birth"] = float(
                    np.max(births) - np.min(births)
                )
                features[sample_key][f"dim_{dim}_range_death"] = float(
                    np.max(deaths) - np.min(deaths)
                )
                features[sample_key][f"dim_{dim}_skew_birth"] = safe_skew(births)
                features[sample_key][f"dim_{dim}_kurt_birth"] = safe_kurtosis(births)
                features[sample_key][f"dim_{dim}_skew_death"] = safe_skew(deaths)
                features[sample_key][f"dim_{dim}_kurt_death"] = safe_kurtosis(deaths)

                # Birth-death correlation
                features[sample_key][f"dim_{dim}_birth_death_corr"] = safe_corr(
                    births, deaths
                )

                # Persistence center of mass & distance to diagonal
                mean_birth = features[sample_key][f"dim_{dim}_mean_birth_time"]
                mean_death = features[sample_key][f"dim_{dim}_mean_death_time"]
                features[sample_key][f"dim_{dim}_COM_birth"] = mean_birth
                features[sample_key][f"dim_{dim}_COM_death"] = mean_death
                features[sample_key][f"dim_{dim}_COM_diag_dist"] = float(
                    mean_death - mean_birth
                )

                # Carlsson features
                f_vals = compute_carlsson_features(births, deaths, fn_max=fn_max)
                for j, fv in enumerate(f_vals, start=1):
                    features[sample_key][f"dim_{dim}_carlsson_f{j}"] = float(fv)

            else:
                # Empty diagram → set everything to 0
                for metric in [
                    "max_lifetime",
                    "mean_lifetime",
                    "median_lifetime",
                    "std_lifetime",
                    "variance_lifetime",
                    "min_lifetime",
                    "skew_lifetime",
                    "kurt_lifetime",
                    "IQR_lifetime",
                    "CV_lifetime",
                    "Gini_lifetime",
                    "Q90_lifetime",
                    "Q95_lifetime",
                    "Q99_lifetime",
                    "sum_birth_times",
                    "mean_birth_time",
                    "sum_death_times",
                    "mean_death_time",
                    "range_birth",
                    "range_death",
                    "skew_birth",
                    "kurt_birth",
                    "skew_death",
                    "kurt_death",
                    "birth_death_corr",
                    "COM_birth",
                    "COM_death",
                    "COM_diag_dist",
                    "max_lifetime_ratio",
                ]:
                    features[sample_key][f"dim_{dim}_{metric}"] = 0.0

                for k in [1, 3, 5]:
                    features[sample_key][f"dim_{dim}_top{k}_lifetime_sum"] = 0.0
                    features[sample_key][f"dim_{dim}_top{k}_lifetime_share"] = 0.0

                for j in range(1, fn_max + 1):
                    features[sample_key][f"dim_{dim}_carlsson_f{j}"] = 0.0

    features_df = pd.DataFrame.from_dict(features, orient="index")
    features_df.reset_index(drop=True, inplace=True)
    return features_df


# -------------------------------------------------------------------
# Main: build full windowed TDA feature matrix
# -------------------------------------------------------------------
def build_windowed_tda_feature_matrix(scaled_diagrams, heat_sigma=0.1, heat_bins=100, fn_max=5):
    """
    scaled_diagrams: array-like of persistence diagrams per window
    Returns:
        final_combined_df: DataFrame of all TDA features per window
    """

    # 1) Initialize feature extractors
    persistence_entropy = PersistenceEntropy()
    amplitude_bottleneck = Amplitude(metric="bottleneck")
    amplitude_wasserstein = Amplitude(metric="wasserstein")
    amplitude_landscape = Amplitude(metric="landscape")
    betti_curve = BettiCurve()
    persistence_landscape = PersistenceLandscape()
    silhouette = Silhouette()
    heat_kernel = HeatKernel(sigma=heat_sigma, n_bins=heat_bins, n_jobs=-1)

    # 2) Transform diagrams
    entropy_features = persistence_entropy.fit_transform(scaled_diagrams)
    amplitude_bottleneck_features = amplitude_bottleneck.fit_transform(scaled_diagrams)
    amplitude_wasserstein_features = amplitude_wasserstein.fit_transform(scaled_diagrams)
    amplitude_landscape_features = amplitude_landscape.fit_transform(scaled_diagrams)
    betti_features = betti_curve.fit_transform(scaled_diagrams)
    landscape_features = persistence_landscape.fit_transform(scaled_diagrams)
    silhouette_features = silhouette.fit_transform(scaled_diagrams)
    heat_kernel_features = heat_kernel.fit_transform(scaled_diagrams)

    # 3) Build DataFrames from each group

    # 3.1 Betti curves, landscapes, silhouettes → rich summaries
    betti_df = compute_curve_features(betti_features, prefix="betti")
    landscape_df = compute_curve_features(landscape_features, prefix="landscape")
    silhouette_df = compute_curve_features(silhouette_features, prefix="silhouette")

    # 3.2 Flattenable features: entropy and amplitudes
    entropy_df = flatten_features_to_df(entropy_features, prefix="entropy")
    amplitude_bottleneck_df = flatten_features_to_df(
        amplitude_bottleneck_features, prefix="amplitude_bottleneck"
    )
    amplitude_wasserstein_df = flatten_features_to_df(
        amplitude_wasserstein_features, prefix="amplitude_wasserstein"
    )
    amplitude_landscape_df = flatten_features_to_df(
        amplitude_landscape_features, prefix="amplitude_landscape"
    )

    # 3.3 Heat kernel features (compressed)
    heat_kernel_df = extract_heat_kernel_features(heat_kernel_features)

    # 4) Combine all "simple" features
    combined_df = pd.concat(
        [
            entropy_df,
            amplitude_bottleneck_df,
            amplitude_wasserstein_df,
            amplitude_landscape_df,
            betti_df,
            landscape_df,
            silhouette_df,
            heat_kernel_df,
        ],
        axis=1,
    )

    # 5) Add diagram statistics (lifetimes, birth/death stats, Carlsson + shape)
    stats_df = extract_diagram_statistics(scaled_diagrams, fn_max=fn_max)

    # 6) Cross-homology features (H1 vs H0)
    cross_df = pd.DataFrame()
    # Lifetime sum ratio H1/H0
    if (
        "dim_1_sum_lifetimes" in stats_df.columns
        and "dim_0_sum_lifetimes" in stats_df.columns
    ):
        cross_df["H1_over_H0_sum_lifetimes"] = safe_ratio(
            stats_df["dim_1_sum_lifetimes"].values,
            stats_df["dim_0_sum_lifetimes"].values,
        )
    else:
        cross_df["H1_over_H0_sum_lifetimes"] = 0.0

    # Max lifetime ratio H1/H0
    if (
        "dim_1_max_lifetime" in stats_df.columns
        and "dim_0_max_lifetime" in stats_df.columns
    ):
        cross_df["H1_over_H0_max_lifetime"] = safe_ratio(
            stats_df["dim_1_max_lifetime"].values,
            stats_df["dim_0_max_lifetime"].values,
        )
    else:
        cross_df["H1_over_H0_max_lifetime"] = 0.0

    # Betti AUC ratio H1/H0 (if available)
    if (
        "betti_AUC_H1" in combined_df.columns
        and "betti_AUC_H0" in combined_df.columns
    ):
        cross_df["H1_over_H0_betti_AUC"] = safe_ratio(
            combined_df["betti_AUC_H1"].values,
            combined_df["betti_AUC_H0"].values,
        )
    else:
        cross_df["H1_over_H0_betti_AUC"] = 0.0

    # Entropy difference H1 - H0 (using entropy_0, entropy_1 if they exist)
    # PersistenceEntropy returns one feature per homology dimension
    if "entropy_0" in entropy_df.columns and "entropy_1" in entropy_df.columns:
        cross_df["entropy_diff_H1_minus_H0"] = (
            entropy_df["entropy_1"].values - entropy_df["entropy_0"].values
        )
    else:
        cross_df["entropy_diff_H1_minus_H0"] = 0.0

    # 7) Final concatenation
    final_combined_df = pd.concat(
        [
            combined_df.reset_index(drop=True),
            stats_df.reset_index(drop=True),
            cross_df.reset_index(drop=True),
        ],
        axis=1,
    )

    # 8) Remove columns with NaN or ±inf anywhere
    mask_good = ~final_combined_df.isin([np.inf, -np.inf]).any() & final_combined_df.notna().all()
    final_combined_df = final_combined_df.loc[:, mask_good]

    return final_combined_df


In [7]:
final_combined_df = build_windowed_tda_feature_matrix(
    scaled_diagrams,
    heat_sigma=0.1,
    heat_bins=30,   # make it smaller for quick tests
    fn_max=5
)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

old tda features version 2025 june

In [ ]:
# @title old version 2025 june do not run
import numpy as np
import pandas as pd
from gtda.diagrams import (
    PersistenceEntropy,
    Amplitude,
    BettiCurve,
    PersistenceLandscape,
    Silhouette,
    HeatKernel,
)

# -------------------------------------------------------------------
# Helper: Carlsson features
# -------------------------------------------------------------------
def compute_carlsson_features(births, deaths, fn_max=5):
    """Compute Carlsson-type features f1..f5 for a single diagram."""
    if len(births) == 0:
        return [0.0] * fn_max

    lifetimes = deaths - births
    d_max = np.max(deaths) if len(deaths) > 0 else 0.0

    f1 = np.sum(births * lifetimes) if fn_max >= 1 else None
    f2 = np.sum((d_max - deaths) * lifetimes) if fn_max >= 2 else None
    f3 = np.sum((births ** 2) * (lifetimes ** 4)) if fn_max >= 3 else None
    f4 = np.sum(((d_max - deaths) ** 2) * (lifetimes ** 4)) if fn_max >= 4 else None
    f5 = np.max(lifetimes) if (fn_max >= 5 and len(lifetimes) > 0) else None

    values = [f1, f2, f3, f4, f5][:fn_max]
    return [v if v is not None else 0.0 for v in values]


# -------------------------------------------------------------------
# Helper: heat kernel feature extraction
# -------------------------------------------------------------------
def extract_heat_kernel_features(heat_kernel_features):
    """
    heat_kernel_features: array of shape
        (n_windows, 2, n_bins, n_bins) for H0 and H1

    Returns:
        DataFrame (n_windows, many_features)
    """
    feature_list = []

    for i in range(heat_kernel_features.shape[0]):  # per window
        sample_feats = {}

        for dim_idx, hom_dim in enumerate(["H0", "H1"]):
            matrix = heat_kernel_features[i, dim_idx, :, :]  # shape: (n_bins, n_bins)

            # Global L1/L2
            sample_feats[f"HK_L1_{hom_dim}"] = np.sum(np.abs(matrix))
            sample_feats[f"HK_L2_{hom_dim}"] = np.sqrt(np.sum(matrix ** 2))

            # Per-column L1/L2
            n_cols = matrix.shape[1]
            for col in range(n_cols):
                col_data = matrix[:, col]
                sample_feats[f"HK_Col{col}_L1_{hom_dim}"] = np.sum(np.abs(col_data))
                sample_feats[f"HK_Col{col}_L2_{hom_dim}"] = np.sqrt(
                    np.sum(col_data ** 2)
                )

        feature_list.append(sample_feats)

    heat_kernel_df = pd.DataFrame(feature_list)
    return heat_kernel_df


# -------------------------------------------------------------------
# Helper: Betti / landscape / silhouette norm features
# -------------------------------------------------------------------
def compute_curve_norm_features(curve_array, prefix):
    """
    curve_array: shape (n_windows, 2, n_points)   (H0, H1)
    prefix: string prefix for column names (e.g. 'betti', 'landscape', 'silhouette')

    Returns DataFrame with columns:
        {prefix}_L1_H0, {prefix}_L2_H0, {prefix}_L1_H1, {prefix}_L2_H1
    """
    n_windows = curve_array.shape[0]

    l1_h0, l2_h0, l1_h1, l2_h1 = [], [], [], []

    for i in range(n_windows):
        h0 = curve_array[i, 0, :]
        h1 = curve_array[i, 1, :]

        l1_h0.append(np.sum(np.abs(h0)))
        l2_h0.append(np.sqrt(np.sum(h0 ** 2)))

        l1_h1.append(np.sum(np.abs(h1)))
        l2_h1.append(np.sqrt(np.sum(h1 ** 2)))

    df = pd.DataFrame(
        {
            f"{prefix}_L1_H0": l1_h0,
            f"{prefix}_L2_H0": l2_h0,
            f"{prefix}_L1_H1": l1_h1,
            f"{prefix}_L2_H1": l2_h1,
        }
    )
    return df


# -------------------------------------------------------------------
# Helper: flatten simple vector features
# -------------------------------------------------------------------
def flatten_features_to_df(features_array, prefix):
    """
    features_array: np.ndarray of shape (n_windows, ...)
    prefix: base name for columns
    """
    flattened = features_array.reshape(features_array.shape[0], -1)
    cols = [f"{prefix}_{i}" for i in range(flattened.shape[1])]
    return pd.DataFrame(flattened, columns=cols)


# -------------------------------------------------------------------
# Helper: diagram statistics + Carlsson features
# -------------------------------------------------------------------
def extract_diagram_statistics(scaled_diagrams, fn_max=5):
    """
    scaled_diagrams: array-like of diagrams, each of shape (n_points, 3)
                     columns: [birth, death, homology_dim]
    Returns:
        features_df: DataFrame with per-window summary stats + Carlsson features.
    """
    n_windows = len(scaled_diagrams)
    features = {}

    for dim in [0, 1]:  # H0, H1
        # filter diagrams by dimension
        filtered_diagrams = [
            diagram[diagram[:, 2] == dim] for diagram in scaled_diagrams
        ]

        for i, diagram_dim in enumerate(filtered_diagrams):
            lifetimes = diagram_dim[:, 1] - diagram_dim[:, 0] if len(diagram_dim) > 0 else np.array([])
            sample_key = f"sample_{i}"

            if sample_key not in features:
                features[sample_key] = {}

            # Basic stats
            features[sample_key][f"dim_{dim}_num_features"] = len(lifetimes)
            features[sample_key][f"dim_{dim}_sum_lifetimes"] = float(np.sum(lifetimes))

            if len(lifetimes) > 0:
                features[sample_key][f"dim_{dim}_max_lifetime"] = float(np.max(lifetimes))
                features[sample_key][f"dim_{dim}_mean_lifetime"] = float(np.mean(lifetimes))
                features[sample_key][f"dim_{dim}_median_lifetime"] = float(np.median(lifetimes))
                features[sample_key][f"dim_{dim}_std_lifetime"] = float(np.std(lifetimes))
                features[sample_key][f"dim_{dim}_variance_lifetime"] = float(np.var(lifetimes))
                features[sample_key][f"dim_{dim}_min_lifetime"] = float(np.min(lifetimes))

                births = diagram_dim[:, 0]
                deaths = diagram_dim[:, 1]

                features[sample_key][f"dim_{dim}_sum_birth_times"] = float(np.sum(births))
                features[sample_key][f"dim_{dim}_mean_birth_time"] = float(np.mean(births))
                features[sample_key][f"dim_{dim}_sum_death_times"] = float(np.sum(deaths))
                features[sample_key][f"dim_{dim}_mean_death_time"] = float(np.mean(deaths))

                # Carlsson features
                f_vals = compute_carlsson_features(births, deaths, fn_max=fn_max)
                for j, fv in enumerate(f_vals, start=1):
                    features[sample_key][f"dim_{dim}_carlsson_f{j}"] = float(fv)
            else:
                # Empty diagram → set everything to 0
                for metric in [
                    "max_lifetime",
                    "mean_lifetime",
                    "median_lifetime",
                    "std_lifetime",
                    "variance_lifetime",
                    "min_lifetime",
                    "sum_birth_times",
                    "mean_birth_time",
                    "sum_death_times",
                    "mean_death_time",
                ]:
                    features[sample_key][f"dim_{dim}_{metric}"] = 0.0

                for j in range(1, fn_max + 1):
                    features[sample_key][f"dim_{dim}_carlsson_f{j}"] = 0.0

    features_df = pd.DataFrame.from_dict(features, orient="index")
    features_df.reset_index(drop=True, inplace=True)
    return features_df


# -------------------------------------------------------------------
# Main: build full windowed TDA feature matrix
# -------------------------------------------------------------------
def build_windowed_tda_feature_matrix(scaled_diagrams, heat_sigma=0.1, heat_bins=100, fn_max=5):
    """
    scaled_diagrams: array-like of persistence diagrams per window
    Returns:
        final_combined_df: DataFrame of all TDA features per window
    """

    # 1) Initialize feature extractors
    persistence_entropy = PersistenceEntropy()
    amplitude_bottleneck = Amplitude(metric="bottleneck")
    amplitude_wasserstein = Amplitude(metric="wasserstein")
    amplitude_landscape = Amplitude(metric="landscape")
    betti_curve = BettiCurve()
    persistence_landscape = PersistenceLandscape()
    silhouette = Silhouette()
    heat_kernel = HeatKernel(sigma=heat_sigma, n_bins=heat_bins, n_jobs=-1)

    # 2) Transform diagrams
    entropy_features = persistence_entropy.fit_transform(scaled_diagrams)
    amplitude_bottleneck_features = amplitude_bottleneck.fit_transform(scaled_diagrams)
    amplitude_wasserstein_features = amplitude_wasserstein.fit_transform(scaled_diagrams)
    amplitude_landscape_features = amplitude_landscape.fit_transform(scaled_diagrams)
    betti_features = betti_curve.fit_transform(scaled_diagrams)
    landscape_features = persistence_landscape.fit_transform(scaled_diagrams)
    silhouette_features = silhouette.fit_transform(scaled_diagrams)
    heat_kernel_features = heat_kernel.fit_transform(scaled_diagrams)

    # 3) Build DataFrames from each group

    # 3.1 Betti curves, landscapes, silhouettes → norm-based summaries
    betti_df = compute_curve_norm_features(betti_features, prefix="betti")
    landscape_df = compute_curve_norm_features(landscape_features, prefix="landscape")
    silhouette_df = compute_curve_norm_features(silhouette_features, prefix="silhouette")

    # 3.2 Flattenable features: entropy and amplitudes
    entropy_df = flatten_features_to_df(entropy_features, prefix="entropy")
    amplitude_bottleneck_df = flatten_features_to_df(
        amplitude_bottleneck_features, prefix="amplitude_bottleneck"
    )
    amplitude_wasserstein_df = flatten_features_to_df(
        amplitude_wasserstein_features, prefix="amplitude_wasserstein"
    )
    amplitude_landscape_df = flatten_features_to_df(
        amplitude_landscape_features, prefix="amplitude_landscape"
    )

    # 3.3 Heat kernel features
    heat_kernel_df = extract_heat_kernel_features(heat_kernel_features)

    # 4) Combine all "simple" features
    combined_df = pd.concat(
        [
            entropy_df,
            amplitude_bottleneck_df,
            amplitude_wasserstein_df,
            amplitude_landscape_df,
            betti_df,
            landscape_df,
            silhouette_df,
            heat_kernel_df,
        ],
        axis=1,
    )

    # 5) Add diagram statistics (lifetimes, birth/death stats, Carlsson)
    stats_df = extract_diagram_statistics(scaled_diagrams, fn_max=fn_max)
    final_combined_df = pd.concat(
        [combined_df.reset_index(drop=True), stats_df.reset_index(drop=True)], axis=1
    )

    # 6) Remove columns with NaN or ±inf anywhere
    mask_good = ~final_combined_df.isin([np.inf, -np.inf]).any() & final_combined_df.notna().all()
    final_combined_df = final_combined_df.loc[:, mask_good]

    return final_combined_df
## final_combined_df = build_windowed_tda_feature_matrix(scaled_diagrams)

In [ ]:
# @title
#print(list(final_combined_df.columns))

In [ ]:
# @title
##file_path = 'tda_features_from_residuals_all.xlsx'
#final_combined_df.to_excel(file_path, index=False)

FEATURE SELECTION

In [8]:
# Dropping columns with low variance from final_combined_df
if 'final_combined_df' in locals():
    # Calculate variance of each column
    variance = final_combined_df.var()

    # Threshold for low variance (can be adjusted)
    low_variance_threshold = 0.01  # Example: close to zero variance

    # Identify columns with variance below the threshold
    low_variance_columns = variance[variance < low_variance_threshold].index

    # Drop low variance columns
    final_combined_df_cleaned = final_combined_df.drop(columns=low_variance_columns)

In [9]:
data=final_combined_df_cleaned

In [10]:
# Identify columns with inf, -inf, or NaN values
cols_with_invalid_values = data.columns[data.isin([np.inf, -np.inf]).any() | data.isna().any()]

# Replace inf and -inf with NaN first
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill NaN values with column mean
data[cols_with_invalid_values] = data[cols_with_invalid_values].apply(lambda col: col.fillna(col.mean()), axis=0)

In [11]:
simplified_df=data

In [12]:
# Adjust the indices of the feature dataset for regression analysis
regression_features_df = simplified_df[:-1].reset_index(drop=True)
regression_target_series = df['Operation'][window_size:].reset_index(drop=True)

# Combine features and target into a single dataset for regression analysis
regression_dataset = regression_features_df.copy()
regression_dataset['Target'] = regression_target_series

In [13]:
# Adjust the indices of the feature dataset for regression analysis
regression_features_df = simplified_df[:-1].reset_index(drop=True)
regression_target_series = df['Operation'][window_size:].reset_index(drop=True)

# Combine features and target into a single dataset for regression analysis
regression_dataset = regression_features_df.copy()
regression_dataset['Target'] = regression_target_series

In [14]:
df2=regression_dataset

In [15]:
from sklearn.ensemble import RandomForestRegressor
from feature_engine.selection import SmartCorrelatedSelection

In [16]:
# Assume the last column is the target variable (modify if needed)
target_column = "Target"  # Change if your target column has a different name
X = df2.drop(columns=[target_column])  # Extract feature columns
y = df2[target_column]  # Define the target variable

# Initialize SmartCorrelatedSelection for regression
tr = SmartCorrelatedSelection(
    variables=None,  # Apply to all numeric features
    method="pearson",  # Pearson correlation method
    threshold=0.99,  # Correlation threshold
    missing_values="raise",  # Raise error if missing values exist
    selection_method="model_performance",  # Use model performance to select features
    estimator=RandomForestRegressor(random_state=1, n_estimators=100),  # RandomForest for regression
    scoring="r2",  # Use R² as the evaluation metric
    cv=3  # 3-fold cross-validation
)

# Fit and transform the dataset to select best features
X_selected = tr.fit_transform(X, y)

# Reconstruct df2 with selected features and target variable
df2 = pd.concat([X_selected, y], axis=1)

In [17]:
from sklearn.preprocessing import MinMaxScaler

# Separate features and target
features = df2.drop(columns=["Target"])
target = df2["Target"]

# Apply scaling only to features
scaler = MinMaxScaler()
features_scaled = pd.DataFrame(scaler.fit_transform(features), columns=features.columns, index=features.index)

# Recombine scaled features with the original Target
df2_scaled = pd.concat([features_scaled, target], axis=1)

In [18]:
file_path = 'tda_features_from_residuals_selected_scaled.xlsx'
df2_scaled.to_excel(file_path, index=False)  # index=False to exclude the index column

In [19]:
df2_scaled

,entropy_0,entropy_1,amplitude_wasserstein_0,betti_L2_H0,betti_median_H0,betti_std_H0,betti_skew_H0,betti_kurt_H0,betti_AUC_H0,betti_L1_H1,...,dim_0_range_death,dim_0_kurt_death,dim_0_carlsson_f2,dim_0_carlsson_f4,dim_1_mean_death_time,dim_1_range_death,dim_1_kurt_birth,dim_1_kurt_death,entropy_diff_H1_minus_H0,Target
0,0.896208,0.507349,0.443346,0.637952,0.000000,0.734808,0.225990,0.104275,0.496296,0.1875,...,0.304506,0.181997,0.177915,0.008330,0.325599,0.459239,3.330669e-16,2.220446e-16,0.501601,5.605425
1,0.884877,0.507349,0.608247,0.788978,0.333333,0.784898,0.129265,0.060399,0.670370,0.1875,...,0.432599,0.334762,0.400818,0.067037,0.325599,0.459239,3.330669e-16,2.220446e-16,0.506430,7.289203
2,0.759823,0.507349,0.667709,0.701659,0.333333,0.603150,0.166612,0.093081,0.670370,0.1875,...,0.680394,0.510541,0.576087,0.156772,0.325599,0.459239,3.330669e-16,2.220446e-16,0.559719,13.613149
3,0.635374,0.507349,0.795855,0.688754,0.333333,0.478813,0.265729,0.126075,0.722222,0.1875,...,0.962615,0.836989,0.870230,0.317119,0.325599,0.459239,3.330669e-16,2.220446e-16,0.612751,-8.930337
4,0.717890,0.507349,1.000000,0.925372,0.666667,0.430889,0.060005,0.039323,1.000000,0.3750,...,0.962615,0.273957,0.942014,0.766706,0.349509,0.492963,3.330669e-16,3.330669e-16,0.577588,-6.574519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619,0.717985,0.000000,0.194923,0.196413,0.000000,0.285187,0.600863,0.469813,0.181481,0.0000,...,0.244309,0.215253,0.073930,0.000396,0.000000,0.000000,1.000000e+00,1.000000e+00,0.118564,11.793560
620,0.774962,0.507349,0.212006,0.251862,0.000000,0.353686,0.535314,0.383215,0.211111,0.2500,...,0.244309,0.201228,0.078952,0.000595,0.036940,0.052102,3.330669e-16,3.330669e-16,0.553268,3.644472
621,0.655220,0.507349,0.273796,0.257425,0.000000,0.326177,0.514290,0.383861,0.248148,0.2500,...,0.320961,0.192822,0.103049,0.000849,0.036940,0.052102,3.330669e-16,3.330669e-16,0.604294,0.522852
622,0.741534,0.507349,0.314165,0.363711,0.000000,0.449005,0.454492,0.319631,0.311111,0.2500,...,0.277697,0.167772,0.117907,0.001306,0.036940,0.052102,3.330669e-16,3.330669e-16,0.567513,-8.146389


this is only for old version of tda features

In [ ]:
# @title old version of pca of tda features
from sklearn.decomposition import PCA

df = df2_scaled.copy()
hk_cols = [c for c in df.columns if c.startswith("HK_Col")]

X_hk = df[hk_cols].to_numpy()
pca = PCA(n_components=5)  # keep 5 PCs (tune this)
hk_pca = pca.fit_transform(X_hk)

hk_pca_df = pd.DataFrame(
    hk_pca,
    columns=[f"HK_PCA_{i+1}" for i in range(hk_pca.shape[1])],
    index=df.index
)

df_no_hkcols = df.drop(columns=hk_cols)
final_reduced_df = pd.concat([df_no_hkcols, hk_pca_df], axis=1)


In [ ]:
# @title
#final_reduced_df

In [ ]:
# @title
#file_path = 'tda_features_from_residuals_selected_scaled_w_pca.xlsx'
#final_reduced_df .to_excel(file_path, index=False)  # index=False to exclude the index column